# Step 8d: tenure and product diversity
Last two candidates, judged by **rule E** against the kept model (R, F, M + interval + overdue ratio).
- **Tenure** = days from the customer's first purchase inside the 12-month window to the cutoff (capped at 365: it cannot see older history).
- **NProducts** = number of distinct stock codes bought in the window.

In [1]:
import sys, warnings; sys.path.insert(0, '../src'); warnings.filterwarnings('ignore')
import pandas as pd
from step8_features import *
pd.set_option('display.width', 200)
train = make_snapshot(load_ledger(), TRAIN_CUTOFF); y = train.target
base = ('IPI', 'OverdueRatio'); X_kept = build_features(train, base)
build_features(train, ('Tenure', 'NProducts'))[['Tenure', 'NProducts']].describe().round(1)

,Tenure,NProducts
count,4242.0,4242.0
mean,217.7,62.9
std,117.4,84.2
min,0.0,1.0
25%,113.0,17.0
50%,244.0,38.0
75%,320.0,78.0
max,364.0,1713.0


In [2]:
results = {}
for extra in (('Tenure',), ('NProducts',), ('Tenure', 'NProducts')):
    t = screen(X_kept, build_features(train, base + extra), y); bt = customer_bootstrap(X_kept, build_features(train, base + extra), y)
    results[extra] = (t, bt)
    print('=== +', ' + '.join(extra)); print(summarize(t).round(4).to_string())
    print({k: tuple(round(x, 4) for x in v) for k, v in bt.items()}, '\n')
pd.DataFrame({' + '.join(e): {k.split(':')[0]: ('KEEP' if v else 'reject') for k, v in rule_verdicts(t, bt).items()} for e, (t, bt) in results.items()}).T

=== + Tenure
                  mean      sd  improved       t
R2              0.0169  0.0082      10.0  6.5155
MAE gain (GBP)  2.7542  1.9090       9.0  4.5625
Capture@20%     0.0015  0.0009      10.0  5.4459
{'R2': (0.0172, -0.0037, 0.0441), 'MAE gain (GBP)': (2.2111, -3.0463, 8.9588), 'Capture@20%': (0.0022, -0.0018, 0.0066)} 



=== + NProducts
                  mean      sd  improved       t
R2              0.0020  0.0107       7.0  0.5939
MAE gain (GBP) -0.4756  2.5405       5.0 -0.5919
Capture@20%    -0.0003  0.0009       5.0 -0.9830
{'R2': (0.0009, -0.0101, 0.008), 'MAE gain (GBP)': (-0.2123, -3.105, 2.8153), 'Capture@20%': (0.0002, -0.0025, 0.0035)} 



=== + Tenure + NProducts
                  mean      sd  improved       t
R2              0.0158  0.0140      10.0  3.5540
MAE gain (GBP)  1.4144  2.7817       7.0  1.6079
Capture@20%     0.0015  0.0011      10.0  4.3775
{'R2': (0.0122, -0.0032, 0.0423), 'MAE gain (GBP)': (-0.1763, -3.8514, 3.9399), 'Capture@20%': (0.0009, -0.0036, 0.0051)} 



,A,B,C,D,E
Tenure,KEEP,KEEP,KEEP,reject,reject
NProducts,reject,reject,reject,reject,reject
Tenure + NProducts,KEEP,KEEP,KEEP,reject,reject


## What we learned
- **Tenure** is the nearest miss: R² +0.017 (interval -0.004 to +0.044), better in 10/10 fold splits, but the interval touches zero and MAE is not clearly better (+2.2, -3.0 to +9.0). Rule E rejects it.
- **NProducts** adds nothing (R² +0.001, MAE -0.2).
- Both: rejected.
- **End of the feature list.** Of 7 candidate columns tried (AOV, IPI, OverdueRatio, RecentShare, ReturnRate, Tenure, NProducts) only IPI + OverdueRatio survive, as a provisional group.